# Which eigenvalues drive the collective response, and at which frequency?

Every pair $(G_1, G_2)$ is matched on $\lambda_2$, $\lambda_{max}$ and edge count, so the two
ends of the spectrum are pinned and only the **bulk** differs. Within a pair, any change in
collective response $H^2$ therefore has to come from the bulk. Correlating "how far did
eigenvalue $i$ move" against "how much did $H^2$ change at frequency $\omega$", across many
pairs, localises which eigenvalues the response cares about and at which driving frequency.

**No binning.** Each of the 239 usable eigenvalues is its own row. The graphs have $n=240$
nodes and therefore 240 Laplacian eigenvalues, but $\lambda_1 = 0$ exactly for a connected
graph (the constant eigenvector; it appears as $\sim 10^{-14}$ in the files). The analysis
uses *relative* differences $(\lambda^{(2)}_i - \lambda^{(1)}_i)/\lambda^{(1)}_i$, which is
$0/0$ for that mode, so it is dropped and 239 remain. Earlier versions of this notebook also
swept a "region" binning (30/40/50/60/120 contiguous blocks of the spectrum) inherited from
the cospectral analysis; that is gone -- one eigenvalue per row is finer than any of them.

**Frequency is on the $x$ axis** in the main figures: a horizontal slice is the frequency
response of one eigenvalue, a vertical slice is the state of the whole spectrum at one
frequency.

Two reference families, analysed separately and then compared: **ws** (Watts-Strogatz,
$k=16$, homogeneous) and **mhk** (Modified Holme-Kim, $k=16$, hub-dominated), 14\,000 pairs
each.


In [ ]:
import os
import shutil
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.cm import ScalarMappable
from matplotlib.colors import TwoSlopeNorm, Normalize
from scipy.stats import rankdata, pearsonr, spearmanr, t as tdist

# ---- inputs, written by spec_div_csv_gen.py (one pair of CSVs per family)
SOURCES = {
    "ws": ("all_graph_eigenvalues_ws.csv", "all_graphs_h2_data_ws.csv"),
    "mhk": ("all_graph_eigenvalues_mhk.csv", "all_graphs_h2_data_mhk.csv"),
}

# The pair matching pins lambda_2 and lambda_max of this same spectrum, so the control
# quantities are just its first and last usable columns.
CONTROL_SOURCES = None

# ---- data/output locations (release layout; notebook runs from scripts/spec_div/)
DATA_DIR = os.environ.get("SPEC_DIV_DATA", os.path.join("..", "..", "nets", "spec_div"))
OUT_DIR = os.environ.get("SPEC_DIV_OUT", os.path.join("..", "..", "output", "spec_div"))
SOURCES = {f: tuple(os.path.join(DATA_DIR, p) for p in v) for f, v in SOURCES.items()}
if CONTROL_SOURCES:
    CONTROL_SOURCES = {f: os.path.join(DATA_DIR, p) for f, p in CONTROL_SOURCES.items()}

# ---- family selection
# SPEC_DIV_FAMILY controls which network family this run analyses:
#   "both" (default)  -> ws and mhk loaded into one frame and drawn as side-by-side
#                        column blocks in a shared figure set. Original behaviour,
#                        kept so the existing figs/ and figs_norm/ still reproduce.
#   "ws" / "mhk"      -> that family ALONE, into its own figure directory. Nothing
#                        downstream sees the other family: FAMS has one entry, so the
#                        per-family panels collapse to a single column and the
#                        cross-family comparison cell skips itself. No pooled
#                        statistic is computed anywhere in such a run.
FAMILY = os.environ.get("SPEC_DIV_FAMILY", "both").strip().lower()
if FAMILY not in ("both", "ws", "mhk"):
    raise ValueError(f"SPEC_DIV_FAMILY must be both|ws|mhk, got {FAMILY!r}")
if FAMILY != "both":
    SOURCES = {FAMILY: SOURCES[FAMILY]}
    if CONTROL_SOURCES:
        CONTROL_SOURCES = {FAMILY: CONTROL_SOURCES[FAMILY]}

# ---- publication figures
# SPEC_DIV_PAPER_FIGS=1 strips the descriptive text that belongs in a journal
# caption rather than on the artwork -- fig.suptitle() and every ax.set_title()
# are suppressed -- and replaces it with a bold panel letter above the top-left
# corner of each panel, matching the style of Fig. 1 of the paper. Axis labels,
# tick labels, legends and colour-bar labels are untouched.
#
# SPEC_DIV_PANEL_OFFSET=second continues the letter sequence instead of
# restarting it. Each of these figures is used in the paper as one half of a
# two-family figure, so the WS half is lettered (a), (b), ... and the MHK half
# picks up where it left off -- (c), (d), ... for a two-panel figure, (e)-(h)
# for the four-panel control. Both halves have the same layout, so the offset is
# just this figure's own panel count and needs no per-figure configuration.
#
# Output goes to paper_figures/ so the annotated working figures under figs_*/
# are preserved -- they carry the on-plot description that is useful when
# reading the analysis rather than the paper.
PAPER_FIGS = os.environ.get("SPEC_DIV_PAPER_FIGS", "0") not in ("0", "", "false", "False")
PANEL_OFFSET = os.environ.get("SPEC_DIV_PANEL_OFFSET", "").strip().lower()
if PAPER_FIGS:
    from matplotlib.axes import Axes as _Axes
    from matplotlib.figure import Figure as _Figure
    _Figure.suptitle = lambda self, *a, **k: None
    _Axes.set_title = lambda self, *a, **k: None


def _is_colorbar(ax):
    return (ax.get_label() == "<colorbar>"
            or hasattr(ax, "_colorbar")
            or getattr(ax, "_colorbar_info", None) is not None)


def add_panel_letters(fig):
    """Bold (a), (b), ... above the top-left of each non-colour-bar panel.

    fig.axes is in creation order, which for these figures is reading order;
    colour-bar axes are interleaved with it and are skipped.
    """
    if not PAPER_FIGS:
        return
    axes = [ax for ax in fig.axes if not _is_colorbar(ax)]
    off = len(axes) if PANEL_OFFSET == "second" else 0
    for i, ax in enumerate(axes):
        ax.text(0.0, 1.02, f"({chr(97 + off + i)})", transform=ax.transAxes,
                ha="left", va="bottom", fontweight="bold", fontsize=18)

FIG_ROOT = "figs" if FAMILY == "both" else f"figs_{FAMILY}"
if PAPER_FIGS:
    FIG_ROOT = "paper_figures/" + FAMILY

FIG_ROOT = os.path.join(OUT_DIR, FIG_ROOT)

SPEC_LABEL = "unnormalised Laplacian $L = D - A$"
SPEC_SHORT = "unnormalised"
LAM_FMT = ".1f"        # lambda in [0.03, 103] here
SPECTRUM_LOGY = True   # index -> lambda spans three decades

# Set to ../spec_div to bring the six synthetic reference topologies into the same
# maps.  Their CSVs are the untagged ones, hence the different names.
COMPARE_WITH_SPEC_DIV = None      # e.g. "../spec_div"

# ---- analysis parameters
min_freq = 0.0001      # lowest frequency to analyse
max_freq = 0.33        # highest frequency to analyse
exclude_freqs = []     # cospectral dropped [0.0210490414451202]; nothing to drop here

# eigenvalues singled out in the readable line figures
PICK_EIGS = [1, 25, 50, 75, 100, 125, 150, 175, 200, 225, 239]

os.makedirs(FIG_ROOT, exist_ok=True)

VERBOSE = True

def log(*args):
    if VERBOSE:
        print(*args, flush=True)

# usetex only if a LaTeX binary is actually on PATH; mathtext renders the same labels
USE_TEX = shutil.which("latex") is not None
plt.rc("text", usetex=USE_TEX)
if USE_TEX:
    plt.rc("text.latex", preamble=r"\usepackage{mathptmx}")
plt.rcParams.update({"axes.labelsize": 15, "xtick.labelsize": 13,
                      "ytick.labelsize": 13})

all_colors = ["blue", "green", "red", "purple", "orange", "brown", "teal", "magenta"]
log("usetex:", USE_TEX, "| spectrum:", SPEC_LABEL, "| output:", FIG_ROOT)


In [ ]:
# Load both families, pair up G1/G2, form the per-pair relative eigenvalue differences.
t0 = time.time()

# G{pair}_{which}_{seed} repeats across families and against ../spec_div, so every row
# is qualified by its source and pairs are keyed on (source, seed, pair).
sources = dict(SOURCES)
if COMPARE_WITH_SPEC_DIV:
    sources["spec_div"] = (f"{COMPARE_WITH_SPEC_DIV}/all_graph_eigenvalues.csv",
                           f"{COMPARE_WITH_SPEC_DIV}/all_graphs_h2_data.csv")

eig_parts, h2_parts = [], []
for source, (eig_path, h2_path) in sources.items():
    if not (os.path.exists(eig_path) and os.path.exists(h2_path)):
        log(f"skipping {source}: {eig_path} not found -- run spec_div_csv_gen.py first")
        continue
    e = pd.read_csv(eig_path)
    h = pd.read_csv(h2_path)
    e["source"] = source
    e["ID"] = source + ":" + e["ID"].astype(str)
    h["ID"] = source + ":" + h["ID"].astype(str)
    eig_parts.append(e)
    h2_parts.append(h)
    log(f"  {source}: {len(e)} graphs, {len(h)} H2 rows")

if not eig_parts:
    raise FileNotFoundError("no CSVs found -- run spec_div_csv_gen.py --family ws/mhk")
eig_df = pd.concat(eig_parts, ignore_index=True)
h2_df = pd.concat(h2_parts, ignore_index=True)
log(f"Loaded {len(eig_df)} eigenvalue rows and {len(h2_df)} H2 rows in {time.time() - t0:.0f}s")

parts = eig_df["ID"].astype(str).str.extract(r"^([^:]+):G(\d+)_([12])_(\d+)$")
parts.columns = ["source", "pair", "which", "seed"]
good = parts.notna().all(axis=1)
if not good.all():
    log(f"dropping {int((~good).sum())} row(s) with an unexpected ID")
eig_df = eig_df[good].reset_index(drop=True)
parts = parts[good].reset_index(drop=True)

h2_wide = h2_df.pivot_table(index="ID", columns="freq", values="H2", aggfunc="last")
h2_pos = {ident: r for r, ident in enumerate(h2_wide.index)}
row_of = {ident: r for r, ident in enumerate(eig_df["ID"])}

by_key = defaultdict(dict)
for ident, src, p, which, sd in zip(eig_df["ID"], parts["source"], parts["pair"],
                                    parts["which"], parts["seed"]):
    by_key[(src, int(sd), int(p))][which] = ident

pair_keys, id1, id2 = [], [], []
for key in sorted(by_key):
    g = by_key[key]
    if "1" in g and "2" in g and g["1"] in h2_pos and g["2"] in h2_pos:
        pair_keys.append(key)
        id1.append(g["1"])
        id2.append(g["2"])
seeds = np.array([k[1] for k in pair_keys])
log(f"Found {len(pair_keys)} complete pairs over {len(set(seeds))} seeds")

base_of = dict(zip(eig_df["ID"], eig_df["base_type"].astype(str)))
base_type = np.array([base_of[i] for i in id1])

eig_cols = [c for c in eig_df.columns if c.startswith("eigenvalue_")]
E = eig_df[eig_cols].to_numpy(dtype=float)
E1, E2 = E[[row_of[i] for i in id1]], E[[row_of[i] for i in id2]]

# lambda_1 = 0 for a connected graph, so its relative difference is 0/0; the
# >1e-10 test is what removes it, leaving 239 of the 240 eigenvalues.
keep = (E1 > 1e-10) & (E2 > 1e-10)
counts = keep.sum(1)
n_eig = int(np.bincount(counts).argmax())
uniform = counts == n_eig
if not uniform.all():
    log(f"dropping {int((~uniform).sum())} pair(s) not having the modal {n_eig}")
    E1, E2, keep = E1[uniform], E2[uniform], keep[uniform]
    seeds, base_type = seeds[uniform], base_type[uniform]
    id1 = [i for i, k in zip(id1, uniform) if k]
    id2 = [i for i, k in zip(id2, uniform) if k]

e1 = E1[keep].reshape(len(E1), n_eig)
e2 = E2[keep].reshape(len(E2), n_eig)
eig_diffs = (e2 - e1) / e1

# The controls are the two eigenvalues the pair matching pinned, which are lambda_2 and
# lambda_max of the UNNORMALISED Laplacian.  When that is also the spectrum being
# analysed they are the first and last columns above; otherwise they have to come from
# the unnormalised CSVs, aligned on the same IDs.
if CONTROL_SOURCES:
    ctl = pd.concat([pd.read_csv(p).assign(ID=lambda d, s=s: s + ":" + d["ID"].astype(str))
                     for s, p in CONTROL_SOURCES.items()], ignore_index=True)
    ctl_cols = [c for c in ctl.columns if c.startswith("eigenvalue_")]
    C = np.sort(ctl[ctl_cols].to_numpy(dtype=float), axis=1)
    ctl_row = {ident: r for r, ident in enumerate(ctl["ID"])}
    missing = [i for i in id1 + id2 if i not in ctl_row]
    if missing:
        raise KeyError(f"{len(missing)} pair member(s) absent from the control CSVs, "
                       f"e.g. {missing[0]}")
    C1, C2 = C[[ctl_row[i] for i in id1]], C[[ctl_row[i] for i in id2]]
    ck = (C1 > 1e-10) & (C2 > 1e-10)
    c_eig = int(np.bincount(ck.sum(1)).argmax())
    c1 = C1[ck].reshape(len(C1), c_eig)
    c2 = C2[ck].reshape(len(C2), c_eig)
    ctl_diffs = (c2 - c1) / c1
    log(f"controls from the unnormalised spectrum ({c_eig} usable eigenvalues)")
else:
    ctl_diffs = eig_diffs
lambda2_diff = ctl_diffs[:, 0]
lambdan_diff = ctl_diffs[:, -1]

log(f"{eig_diffs.shape[0]} pairs x {n_eig} usable eigenvalues "
    f"(240 minus the zero mode)")
log(f"lambda range {e1.min():.4g} to {e1.max():.4g}")
log("families: " + ", ".join(f"{b}={int((base_type == b).sum())}"
                             for b in sorted(set(base_type))))


In [ ]:
# Relative H2 differences on the analysed frequency window
freqs_all = h2_wide.columns.to_numpy(dtype=float)
fmask = (freqs_all >= min_freq) & (freqs_all <= max_freq)
if exclude_freqs:
    fmask &= ~np.isin(np.round(freqs_all, 16), np.round(np.asarray(exclude_freqs), 16))
frequencies = freqs_all[fmask]

H = h2_wide.to_numpy()[:, fmask]
h2_g1 = H[[h2_pos[i] for i in id1]]
h2_g2 = H[[h2_pos[i] for i in id2]]
h2_rel = (h2_g2 - h2_g1) / h2_g1

N_PAIRS = h2_rel.shape[0]
CI95 = 1.96 / np.sqrt(N_PAIRS - 3)
log(f"{N_PAIRS} pairs x {h2_rel.shape[1]} frequencies "
    f"({frequencies.min():.2e} to {frequencies.max():.2e})")
log(f"95% CI half-width, pooled: +/-{CI95:.4f}")


In [ ]:
# Vectorised correlation core.  Spearman is Pearson on ranks, and Pearson between every
# column of X and every column of Y is one standardised matrix product, so the whole
# (eigenvalue x frequency) grid is a single matmul.

def zscore(A):
    A = A - A.mean(0)
    s = A.std(0)
    return A / np.where(s == 0, np.inf, s)

def corr_all(X, Y):
    return zscore(X).T @ zscore(Y) / X.shape[0]

def ranks(A):
    try:
        return rankdata(A, axis=0).astype(float)
    except TypeError:                      # scipy < 1.10 has no axis argument
        return np.column_stack([rankdata(A[:, j]) for j in range(A.shape[1])])

def pvalues(r, n):
    r = np.clip(r, -1 + 1e-15, 1 - 1e-15)
    return 2 * tdist.sf(np.abs(r) * np.sqrt((n - 2) / (1 - r ** 2)), n - 2)

def partial_corr(r_xy, r_xz, r_yz):
    # X vs Y with the control Z removed from both
    num = r_xy - np.outer(r_xz, r_yz)
    den = np.sqrt(np.outer(1 - r_xz ** 2, 1 - r_yz ** 2))
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(den > 1e-12, num / den, np.nan)

def residualise(A, Z):
    Z1 = np.column_stack([np.ones(len(Z)), Z])
    beta, *_ = np.linalg.lstsq(Z1, A, rcond=None)
    return A - Z1 @ beta

FAMS = [f for f in ("ws", "mhk", "spec_div") if (base_type == f).any()]
fam_mask = {f: base_type == f for f in FAMS}

h2_ranks = ranks(h2_rel)
lam2_rank = ranks(lambda2_diff[:, None])
r_yz = corr_all(h2_ranks, lam2_rank).ravel()

# per family: the (eigenvalue x frequency) correlation, and the same with the residual
# lambda_2 movement partialled out of both sides
R, R_partial, CI = {}, {}, {}
for f in FAMS:
    m = fam_mask[f]
    Xr = ranks(eig_diffs[m])
    Yr = ranks(h2_rel[m])
    R[f] = corr_all(Xr, Yr)
    z2 = ranks(lambda2_diff[m][:, None])
    R_partial[f] = partial_corr(R[f], corr_all(Xr, z2).ravel(),
                                corr_all(Yr, z2).ravel())
    CI[f] = 1.96 / np.sqrt(int(m.sum()) - 3)
    log(f"{f:<4} {int(m.sum()):>6} pairs  95% CI +/-{CI[f]:.4f}  "
        f"peak |r_s| {np.nanmax(np.abs(R[f])):.3f}  "
        f"lambda range {e1[m].min():.4g}-{e1[m].max():.4g}")

R_pooled = corr_all(ranks(eig_diffs), h2_ranks)


## Validation


In [ ]:
# The vectorised path against scipy, on random (eigenvalue, frequency) cells
rng = np.random.default_rng(0)
worst_s = worst_p = 0.0
for _ in range(12):
    f = FAMS[rng.integers(len(FAMS))]
    m = fam_mask[f]
    i = rng.integers(n_eig)
    j = rng.integers(len(frequencies))
    worst_s = max(worst_s, abs(spearmanr(eig_diffs[m][:, i], h2_rel[m][:, j]).statistic
                               - R[f][i, j]))
    worst_p = max(worst_p, abs(pearsonr(zscore(eig_diffs[m])[:, i],
                                        zscore(h2_rel[m])[:, j]).statistic
                               - corr_all(eig_diffs[m][:, [i]], h2_rel[m][:, [j]])[0, 0]))
log(f"vectorised vs scipy over 12 random cells: "
    f"Spearman max |diff| {worst_s:.2e}, Pearson {worst_p:.2e}")
assert worst_s < 1e-10 and worst_p < 1e-10


def save(fig, path):
    add_panel_letters(fig)
    fig.savefig(f"{path}.png", dpi=600, bbox_inches="tight")
    fig.savefig(f"{path}.pdf", dpi=600, bbox_inches="tight")
    plt.close(fig)

def nearest(freq):
    return int(np.argmin(np.abs(frequencies - freq)))


## Frequency on the $x$ axis


In [ ]:
# THE figure: frequency on x, one row per eigenvalue, one column block per family
idx = np.arange(1, n_eig + 1)
fig, axes = plt.subplots(2, len(FAMS), figsize=(8 * len(FAMS), 9.5), dpi=600,
                         squeeze=False)

for c, f in enumerate(FAMS):
    ax = axes[0, c]
    mesh = ax.pcolormesh(frequencies, idx, R[f], cmap="RdBu_r",
                         norm=TwoSlopeNorm(0, -1, 1), shading="nearest")
    ax.set_xscale("log")
    ax.set_xlabel(r"Frequency $(\omega)$")
    ax.set_title(f"{f}: every eigenvalue against $H^2$ "
                 f"({int(fam_mask[f].sum())} pairs)", fontsize=11)
    fig.colorbar(mesh, ax=ax, label="Spearman correlation")

    ax = axes[1, c]
    segs = [np.column_stack([frequencies, R[f][i]]) for i in range(n_eig)]
    lc = LineCollection(segs, cmap="viridis", norm=Normalize(1, n_eig),
                        linewidths=0.6, alpha=0.75)
    lc.set_array(idx)
    ax.add_collection(lc)
    ax.axhline(0, color="black", ls="--", alpha=0.5)
    ax.set_xscale("log")
    ax.set_xlim(frequencies.min(), frequencies.max())
    ax.set_ylim(-1.05, 1.05)
    ax.set_xlabel(r"Frequency $(\omega)$")
    ax.grid(True, which="both", ls="--", alpha=0.5)
    fig.colorbar(ScalarMappable(norm=Normalize(1, n_eig), cmap="viridis"),
                 ax=ax, label="eigenvalue index")

axes[0, 0].set_ylabel("Eigenvalue index (non-zero, ascending)")
axes[1, 0].set_ylabel(r"Spearman correlation with $\Delta H^2$")
fig.suptitle(f"Per-eigenvalue frequency response, no binning "
             f"({n_eig} eigenvalues of 240, {SPEC_LABEL})", fontsize=12)
fig.tight_layout()
save(fig, f"{FIG_ROOT}/eigenvalue_frequency_response")


In [ ]:
# A readable subset: named eigenvalues, frequency by frequency
fig, axes = plt.subplots(1, len(FAMS), figsize=(7 * len(FAMS), 4.8), dpi=600,
                         squeeze=False)
for c, f in enumerate(FAMS):
    ax = axes[0, c]
    m = fam_mask[f]
    for i, p in enumerate(PICK_EIGS):
        lam = e1[m][:, p - 1].mean()
        ax.plot(frequencies, R[f][p - 1], lw=1.4,
                color=plt.cm.viridis(i / (len(PICK_EIGS) - 1)),
                label=rf"$i={p}$ ($\bar\lambda={lam:{LAM_FMT}}$)")
    ax.axhline(0, color="black", ls="--", alpha=0.5)
    ax.fill_between(frequencies, -CI[f], CI[f], color="gray", alpha=0.25, lw=0)
    ax.set_xscale("log")
    ax.set_ylim(-1.05, 1.05)
    ax.set_xlabel(r"Frequency $(\omega)$")
    ax.set_title(f, fontsize=11)
    ax.grid(True, which="both", ls="--", alpha=0.5)
    ax.legend(fontsize=13, ncol=2, loc="lower left")
axes[0, 0].set_ylabel(r"Spearman correlation with $\Delta H^2$")
fig.suptitle("Selected eigenvalues (grey band: 95\% CI around zero)" if USE_TEX
             else "Selected eigenvalues (grey band: 95% CI around zero)", fontsize=12)
fig.tight_layout()
save(fig, f"{FIG_ROOT}/eigenvalue_frequency_selected")


## ws against mhk


In [ ]:
# ws against mhk on the eigenvalue INDEX axis.
#
# Index i is a different lambda in each family, but i/239 is the same quantile of the
# spectrum in both, is identical for every pair, and involves no averaging.  A
# lambda/lambda_max axis looks more physical but is a pair-dependent, many-to-one
# aggregation: at omega=1e-3 the bin holding mhk's eigenvalue 100 draws from indices
# 41-221 across pairs, straddles the sign change near index 190, and comes out at
# -0.75 where the eigenvalue itself sits at +0.83.  Hence: compare on the index.
if len(FAMS) >= 2:
    a, b = FAMS[0], FAMS[1]
    diff = R[a] - R[b]
    fig, axes = plt.subplots(1, 3, figsize=(19, 5), dpi=600)
    for ax, (data, title, cmap, lim) in zip(axes, [
            (R[a], a, "RdBu_r", 1.0),
            (R[b], b, "RdBu_r", 1.0),
            (diff, f"{a} $-$ {b}", "PuOr", float(np.nanmax(np.abs(diff))))]):
        mesh = ax.pcolormesh(frequencies, idx, data, cmap=cmap,
                             norm=TwoSlopeNorm(0, -lim, lim), shading="nearest")
        ax.set_xscale("log")
        ax.set_xlabel(r"Frequency $(\omega)$")
        ax.set_title(title, fontsize=11)
        fig.colorbar(mesh, ax=ax,
                     label="Spearman" if cmap == "RdBu_r" else "difference")
    axes[0].set_ylabel("Eigenvalue index (non-zero, ascending)")
    fig.suptitle(f"{a} against {b}, eigenvalue by eigenvalue ({SPEC_SHORT} spectrum)",
                 fontsize=12)
    fig.tight_layout()
    save(fig, f"{FIG_ROOT}/family_comparison_index")

    both = np.isfinite(R[a]) & np.isfinite(R[b])
    log(f"{a} vs {b}: sign agreement {100 * np.mean(np.sign(R[a][both]) == np.sign(R[b][both])):.0f}%"
        f", mean |difference| {np.mean(np.abs(R[a][both] - R[b][both])):.3f}"
        f", maps correlate at {np.corrcoef(R[a][both], R[b][both])[0, 1]:+.3f}")


In [ ]:
# Which eigenvalue leads, and how strongly, frequency by frequency
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), dpi=600)
for f, marker in zip(FAMS, ("o", "s", "^")):
    best = np.argmax(np.abs(R[f]), axis=0)
    peak = R[f][best, np.arange(R[f].shape[1])]
    axes[0].plot(frequencies, best + 1, marker=marker, ms=3, lw=1.3, label=f)
    axes[1].plot(frequencies, np.abs(peak), marker=marker, ms=3, lw=1.3, label=f)
axes[0].set_xscale("log")
axes[0].set_xlabel(r"Frequency $(\omega)$")
axes[0].set_ylabel("Eigenvalue index with largest $|r_s|$")
axes[0].set_ylim(0, n_eig + 1)
axes[0].set_title("Which eigenvalue the response follows")
axes[0].grid(True, ls="--", alpha=0.6)
axes[0].legend()
axes[1].set_xscale("log")
axes[1].set_xlabel(r"Frequency $(\omega)$")
axes[1].set_ylabel(r"$|r_s|$ there")
axes[1].set_ylim(0, 1.05)
axes[1].set_title("How strong that relationship is")
axes[1].grid(True, ls="--", alpha=0.6)
axes[1].legend()
fig.tight_layout()
save(fig, f"{FIG_ROOT}/dominant_eigenvalue_vs_frequency")

# index -> lambda dictionary, and how far each eigenvalue moves
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), dpi=600)
for i, f in enumerate(FAMS):
    m = fam_mask[f]
    axes[0].plot(idx, e1[m].mean(0), lw=1.4, color=all_colors[i], label=f)
    axes[1].plot(idx, np.abs(eig_diffs[m]).mean(0), lw=1.4, color=all_colors[i], label=f)
if SPECTRUM_LOGY:
    axes[0].set_yscale("log")
axes[0].set_xlabel("Eigenvalue index")
axes[0].set_ylabel(r"mean $\lambda_i$")
axes[0].set_title(f"Index $\\rightarrow$ $\\lambda$, per family ({SPEC_SHORT})")
axes[0].grid(True, ls="--", alpha=0.6)
axes[0].legend()
axes[1].set_xlabel("Eigenvalue index")
axes[1].set_ylabel(r"mean $|\Delta\lambda_i / \lambda_i|$")
axes[1].set_title("How far each eigenvalue moves within a pair")
axes[1].grid(True, ls="--", alpha=0.6)
axes[1].legend()
fig.tight_layout()
save(fig, f"{FIG_ROOT}/spectrum_context")


## Controls


In [ ]:
# Control: the matching constraint pins lambda_2 and lambda_max to 1%, but the residual
# still carries signal, so the map is shown with both partialled out.
z2, zn = lambda2_diff, lambdan_diff
Zboth = np.column_stack([z2, zn])
r_h2_lam2 = corr_all(ranks(z2[:, None]), h2_ranks).ravel()
r_h2_lamn = corr_all(ranks(zn[:, None]), h2_ranks).ravel()
Zr = ranks(Zboth)
R_ctrl = corr_all(residualise(ranks(eig_diffs), Zr), residualise(h2_ranks, Zr))
# the bulk is quoted on the same (unnormalised) spectrum as the pins, so "how much
# tighter is the pin than the bulk" is a like-for-like ratio in both notebooks
log(f"|rel d lambda_2| median {np.median(np.abs(z2)):.4f}, "
    f"|rel d lambda_max| median {np.median(np.abs(zn)):.4f}, "
    f"bulk median {np.median(np.abs(ctl_diffs[:, 1:-1])):.4f}")
log(f"Spearman(dH2, d lambda_2) peak {np.abs(r_h2_lam2).max():+.3f}; "
    f"(dH2, d lambda_max) peak {np.abs(r_h2_lamn).max():+.3f}")
log(f"partialling both out: mean |change| {np.nanmean(np.abs(R_pooled - R_ctrl)):.4f}, "
    f"max {np.nanmax(np.abs(R_pooled - R_ctrl)):.4f}")

fig, axes = plt.subplots(2, 2, figsize=(13, 9), dpi=600)
ax = axes[0, 0]
ax.plot(frequencies, r_h2_lam2, color="crimson", lw=1.6, label=r"$\Delta\lambda_2$")
ax.plot(frequencies, r_h2_lamn, color="navy", lw=1.6, label=r"$\Delta\lambda_{max}$")
ax.fill_between(frequencies, -CI95, CI95, color="gray", alpha=0.25, lw=0)
ax.axhline(0, color="black", ls="--", alpha=0.5)
ax.set_xscale("log")
ax.set_xlabel(r"Frequency $(\omega)$")
ax.set_ylabel(r"Spearman with $\Delta H^2$")
ax.set_ylim(-1.05, 1.05)
ax.grid(True, ls="--", alpha=0.6)
ax.legend(fontsize=15)
ax.set_title("What the pinned eigenvalues still explain", fontsize=10)

for ax, data, title, cmap, lim in (
        (axes[0, 1], R_pooled, "pooled, raw", "RdBu_r", 1.0),
        (axes[1, 0], R_ctrl,
         r"pooled, $\Delta\lambda_2$ and $\Delta\lambda_{max}$ removed", "RdBu_r", 1.0),
        (axes[1, 1], R_pooled - R_ctrl, "the change", "PuOr",
         float(np.nanmax(np.abs(R_pooled - R_ctrl))))):
    mesh = ax.pcolormesh(frequencies, idx, data, cmap=cmap,
                         norm=TwoSlopeNorm(0, -lim, lim), shading="nearest")
    ax.set_xscale("log")
    ax.set_xlabel(r"Frequency $(\omega)$")
    ax.set_ylabel("Eigenvalue index")
    ax.set_title(title, fontsize=10)
    fig.colorbar(mesh, ax=ax, label="Spearman" if cmap == "RdBu_r" else "difference")

fig.suptitle(f"Does the matching constraint drive the result? "
             f"(map on the {SPEC_SHORT} spectrum; the pins are always the "
             f"unnormalised $\\lambda_2, \\lambda_{{max}}$)", fontsize=12)
fig.tight_layout()
save(fig, f"{FIG_ROOT}/matched_eigenvalue_effect")


In [ ]:
# The cloud behind the strongest cell of each family
fig, axes = plt.subplots(1, len(FAMS), figsize=(6 * len(FAMS), 4.6), dpi=600,
                         squeeze=False)
for c, f in enumerate(FAMS):
    m = fam_mask[f]
    i, j = np.unravel_index(np.nanargmax(np.abs(R[f])), R[f].shape)
    x, y = eig_diffs[m][:, i], h2_rel[m][:, j]
    xlo, xhi = np.percentile(x, [1, 99])
    ylo, yhi = np.percentile(y, [1, 99])
    ax = axes[0, c]
    ax.hexbin(x, y, gridsize=50, bins="log", cmap="viridis",
              extent=(xlo, xhi, ylo, yhi))
    ax.axhline(0, color="white", lw=0.6, alpha=0.6)
    ax.axvline(0, color="white", lw=0.6, alpha=0.6)
    ax.set_xlim(xlo, xhi)
    ax.set_ylim(ylo, yhi)
    ax.set_xlabel(rf"$\Delta\lambda_{{{i + 1}}}/\lambda_{{{i + 1}}}$")
    ax.set_title(rf"{f}: eigenvalue {i + 1}, $\omega={frequencies[j]:.1e}$, "
                 rf"$r_s={R[f][i, j]:+.2f}$", fontsize=10)
axes[0, 0].set_ylabel(r"$\Delta H^2 / H^2$")
fig.suptitle("The joint distribution behind the strongest correlation "
             "(1-99 percentile window)", fontsize=11)
fig.tight_layout()
save(fig, f"{FIG_ROOT}/scatter_diagnostics")
log("wrote:", sorted(n for n in os.listdir(FIG_ROOT) if n.endswith(".png")))


## Summary


In [ ]:
# Which eigenvalue leads at each frequency, per family
rows = []
for f in FAMS:
    best = np.argmax(np.abs(R[f]), axis=0)
    for target in (1e-4, 1e-3, 1e-2, 1e-1, 3e-1):
        j = nearest(target)
        i = int(best[j])
        rows.append({
            "family": f,
            "omega": f"{frequencies[j]:.2e}",
            "leading eigenvalue": i + 1,
            "its mean lambda": float(f"{e1[fam_mask[f]][:, i].mean():{LAM_FMT}}"),
            "r_s": round(float(R[f][i, j]), 3),
            "r_s, lambda_2 removed": round(float(R_partial[f][i, j]), 3),
            "eigenvalues with |r_s|>0.5": int((np.abs(R[f][:, j]) > 0.5).sum()),
        })
pd.DataFrame(rows).set_index(["family", "omega"])
